# SQLite: query and persist structured data

**Stage:** Advanced · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–15. No variables or files from other notebooks are required.

## What you will learn

- Create tables and parameterized queries
- Aggregate with SQL
- Use transactions and close connections


## Your learning loop

1. Read the explanation and predict the next output.
2. Run the example with **Shift+Enter**.
3. Follow **Try a change**; explain the result in your own words.
4. Complete both exercises and enable their checks.
5. Build the mini-project, then use **Restart Kernel and Run All**.

Examples run as provided. Exercise starters are deliberately unfinished; their checks are disabled until you enable them. A successful Run All with checks disabled does **not** mean the exercises are complete.

Save a copy before extensive experiments. Use one notebook at a time to avoid mixing kernel state.


## 1. A database lifecycle

SQLite is included with Python. ':memory:' creates a temporary database; a filename creates a persistent one. A connection's with block handles transactions but does not close it. closing() ensures cleanup.


In [1]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as connection:
    connection.row_factory = sqlite3.Row
    with connection:
        connection.execute("CREATE TABLE expenses (id INTEGER PRIMARY KEY, category TEXT NOT NULL, amount INTEGER CHECK(amount > 0))")
        connection.executemany("INSERT INTO expenses(category, amount) VALUES (?, ?)",
                               [("food", 120), ("travel", 80), ("food", 50)])
    rows = connection.execute("SELECT category, SUM(amount) AS total FROM expenses GROUP BY category ORDER BY category").fetchall()
    totals = {row["category"]: row["total"] for row in rows}
print(totals)
assert totals == {"food": 170, "travel": 80}


{'food': 170, 'travel': 80}


**Try a change:** Add an expense to the input rows and predict the new totals.


## 2. Parameters are data

Use ? placeholders for values. Concatenating user values into SQL can change the query's meaning. Parameters do not stand in for table or column names; those need a trusted allowlist if dynamic.


In [2]:
with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("CREATE TABLE users (name TEXT)")
    connection.executemany("INSERT INTO users VALUES (?)", [("Mira",), ("Dev",)])
    entered_name = "' OR 1=1 --"
    matches = connection.execute("SELECT name FROM users WHERE name = ?", (entered_name,)).fetchall()
print("Matches:", matches)
assert matches == []


Matches: []


**Try a change:** Set entered_name to Mira. Observe the result without changing the SQL.


## 3. Roll back a failed transaction

A transaction groups changes that should succeed or fail together. An exception escaping the connection's with block rolls back the transaction. Here one invalid row prevents both inserts from being committed.


In [3]:
with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("CREATE TABLE numbers (value INTEGER CHECK(value > 0))")
    try:
        with connection:
            connection.execute("INSERT INTO numbers VALUES (?)", (5,))
            connection.execute("INSERT INTO numbers VALUES (?)", (-1,))
    except sqlite3.IntegrityError:
        print("Both inserts rolled back")
    count = connection.execute("SELECT COUNT(*) FROM numbers").fetchone()[0]
assert count == 0


Both inserts rolled back


**Try a change:** Use 1 instead of -1 and predict count. Update the expected assertion for that experiment.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Filter with SQL

Implement expensive_items(connection, minimum) to return (name, price) rows with price >= minimum, ordered by price then name. The items table already exists in the checks.

<details>
<summary>Show a hint</summary>

Use a one-element tuple (minimum,) for the parameter.

</details>


In [4]:
def expensive_items(connection, minimum):
    pass


In [5]:
exercise_1_ready = False  # Set True when your implementation is ready.
if exercise_1_ready:
    with closing(sqlite3.connect(":memory:")) as db:
        db.execute("CREATE TABLE items (name TEXT, price INTEGER)")
        db.executemany("INSERT INTO items VALUES (?, ?)", [("pen", 10), ("book", 80), ("bag", 200)])
        assert expensive_items(db, 80) == [("book", 80), ("bag", 200)]
        assert expensive_items(db, 999) == []
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.


### Exercise 2: Insert one row

Implement add_item(connection, name, price) with a parameterized INSERT into items(name, price). The caller controls committing.

<details>
<summary>Show a hint</summary>

An apostrophe in a value should work without manual escaping.

</details>


In [6]:
def add_item(connection, name, price):
    pass


In [7]:
exercise_2_ready = False  # Set True when your implementation is ready.
if exercise_2_ready:
    with closing(sqlite3.connect(":memory:")) as db:
        db.execute("CREATE TABLE items (name TEXT, price INTEGER)")
        add_item(db, "Mira's book", 90)
        assert db.execute("SELECT name, price FROM items").fetchall() == [("Mira's book", 90)]
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.


## Build it yourself

Build a task table with id, title, and done. Insert three tasks, mark one complete with UPDATE, and query unfinished tasks. Use a temporary file database and reopen it to demonstrate persistence.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

SQL row order is unspecified without ORDER BY. Remember to commit writes intended to persist. A real application also needs schema migrations and a backup plan.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Worked solutions](../solutions/16_sqlite_and_persistence_solutions.ipynb) · [Previous notebook](15_algorithms_and_complexity.ipynb) · [Next notebook](17_http_apis_and_json.ipynb)
